# Notebook 6 — Expanded ML models: 5-HT₂A & Dopamine D3 (Option B, uniform Landrum)

Rebuilds the **ML models only** (the descriptive chapter stays on the locked Phase I–IV dataset).
For each of the two ML targets we:

1. **Fetch all raw Kᵢ measurements across every phase** (preclinical + Phase I–IV) — one fetch per target.
2. **Curate identically** (salt-strip → MW<700 → peptide removal) and apply the **Landrum 2024**
   artefact filter (drop exact-duplicate copies and Δ=3.0-log unit errors) to the **whole raw set**,
   then take the highest pKᵢ per molecule. This is applied *uniformly* — so 'we applied the paper's
   curation' is literally true for the entire training set.
3. **Report the preclinical vs Phase I–IV split** (transparency on what the preclinical data added).
4. **Fuse ECFP4 + DescriptaStorus**, train **XGBoost + Random Forest**.
5. **Validate two ways: repeated scaffold split AND leave-one-out (LOO).**

> Note: the locked `curated_15desc.csv` cannot be Landrum-curated (it is already deduplicated — the raw
> repeat measurements are gone), which is why the ML set is fetched fresh. The descriptive analysis is
> unaffected. Requires internet for Part 1.

## 0. Config

In [ ]:
import os, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs, RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from descriptastorus.descriptors import rdNormalizedDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, root_mean_squared_error as rmse
try:
    from xgboost import XGBRegressor; HAVE_XGB=True
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor; HAVE_XGB=False
    print('xgboost not found -> using HistGradientBoosting')
TARGETS = {'5-HT2A':'P28223', 'Dopamine D3':'P35462'}   # name -> UniProt
print('boosting model:', 'XGBoost' if HAVE_XGB else 'HistGradientBoosting')

## 1. Fetch ALL raw Kᵢ (every phase) for the two targets
Human, binding (`assay_type='B'`), `standard_type='Ki'`, with a valid pChEMBL value; also attach each
molecule's `max_phase` so we can report the preclinical split later. We keep **all** phases.

In [ ]:
from chembl_webresource_client.new_client import new_client
target_api, activity_api, molecule_api = new_client.target, new_client.activity, new_client.molecule
def resolve(acc):
    h=list(target_api.filter(target_components__accession=acc, target_type='SINGLE PROTEIN',
                             organism='Homo sapiens').only(['target_chembl_id']))
    return h[0]['target_chembl_id'] if h else None
def fetch_all(name, acc):
    tid=resolve(acc); print(f'{name}: {tid}')
    acts=activity_api.filter(target_chembl_id=tid, standard_type='Ki', assay_type='B',
                             pchembl_value__isnull=False, target_organism='Homo sapiens')\
                     .only(['molecule_chembl_id','canonical_smiles','pchembl_value'])
    raw=pd.DataFrame(list(acts))
    ids=raw['molecule_chembl_id'].dropna().unique().tolist(); ph={}
    for i in range(0,len(ids),200):
        for m in molecule_api.filter(molecule_chembl_id__in=ids[i:i+200]).only(['molecule_chembl_id','max_phase']):
            ph[m['molecule_chembl_id']]=m['max_phase']
    raw['max_phase']=pd.to_numeric(raw['molecule_chembl_id'].map(ph), errors='coerce')
    raw['target']=name
    print(f'  {len(raw)} raw Ki measurements, {raw.molecule_chembl_id.nunique()} molecules (all phases)')
    return raw
raw_all = pd.concat([fetch_all(n,a) for n,a in TARGETS.items()], ignore_index=True)

## 2. Curate + apply Landrum **uniformly** to the whole raw set
Salt-strip, drop unparsable SMILES, MW<700, remove peptides. Then, within each (molecule, target),
the Landrum filter: drop exact-duplicate pKᵢ copies, drop values exactly 3.0 log apart (unit errors),
and take the highest remaining pKᵢ (matching the thesis convention Filippo asked for).

In [ ]:
sr=SaltRemover()
pep=Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
def parent(s):
    m=Chem.MolFromSmiles(str(s)) if isinstance(s,str) else None
    if m is None: return None
    p=sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms()==0:
        fr=Chem.GetMolFrags(m,asMols=True); p=max(fr,key=lambda x:x.GetNumAtoms()) if fr else m
    return p
d=raw_all.copy()
d['pKi']=pd.to_numeric(d['pchembl_value'],errors='coerce'); d=d.dropna(subset=['pKi'])
d['mol']=d['canonical_smiles'].apply(parent); d=d[d['mol'].notna()]
d=d[d['mol'].apply(lambda m: Descriptors.MolWt(m))<700]
d=d[~d['mol'].apply(lambda m: m.HasSubstructMatch(pep))]
d['parent_smiles']=d['mol'].apply(Chem.MolToSmiles)

def landrum(v):
    v=pd.Series(v).round(2)
    v=v[~v.duplicated(keep='first')].tolist()               # drop exact-duplicate copies
    v=[x for x in v if not any(abs(abs(x-o)-3.0)<1e-6 for o in v if o!=x)]  # drop 3.0-log unit-error pairs
    return max(v) if v else np.nan                          # highest pKi

g=d.groupby(['molecule_chembl_id','target'])
agg=pd.DataFrame({
    'pKi': g['pKi'].apply(lambda s: landrum(s.tolist())),
    'parent_smiles': g['parent_smiles'].first(),
    'max_phase': g['max_phase'].first(),
}).reset_index().dropna(subset=['pKi'])
agg['preclinical']=~agg['max_phase'].isin([1,2,3,4])
print('curated ML set (uniform Landrum):')
for t in TARGETS:
    s=agg[agg.target==t]
    print(f"  {t}: {len(s)} molecules  ({int(s.preclinical.sum())} preclinical + {int((~s.preclinical).sum())} Phase I-IV)")

## 3. Features: ECFP4 + DescriptaStorus (fused)

In [ ]:
gen=rdNormalizedDescriptors.RDKit2DNormalized()
def featurize(sub):
    sub=sub.reset_index(drop=True)
    mols=[Chem.MolFromSmiles(s) for s in sub.parent_smiles]; keep=[i for i,m in enumerate(mols) if m]
    sub=sub.iloc[keep].reset_index(drop=True); mols=[mols[i] for i in keep]
    N=len(mols); E=np.zeros((N,2048),int)
    for i,m in enumerate(mols): DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m,2,nBits=2048),E[i])
    D=np.nan_to_num(np.array([gen.process(s)[1:] for s in sub.parent_smiles],dtype=float))
    X=np.hstack([E,D]); y=sub.pKi.values.astype(float)
    scaf=np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) for m in mols])
    return X,y,scaf
feat={t:featurize(agg[agg.target==t]) for t in TARGETS}
for t in TARGETS: print(t,'-> X',feat[t][0].shape)

## 4. Train & validate — scaffold split AND LOO

In [ ]:
def boost():
    return XGBRegressor(n_estimators=400,max_depth=4,learning_rate=0.05,subsample=0.8,
                        colsample_bytree=0.6,random_state=0,n_jobs=-1) if HAVE_XGB \
           else HistGradientBoostingRegressor(random_state=0)
def rf(): return RandomForestRegressor(n_estimators=400,random_state=0,n_jobs=-1)
MODELS={('XGB' if HAVE_XGB else 'GB'):boost,'RF':rf}
def scaffold_split(scaf,N,seed,tf=0.2):
    rng=np.random.RandomState(seed); gr={}
    for i,s in enumerate(scaf): gr.setdefault(s,[]).append(i)
    ks=list(gr); rng.shuffle(ks); nt=int(round(tf*N)); te=[]
    for k in ks:
        if len(te)>=nt: break
        te+=gr[k]
    te=set(te); return [i for i in range(N) if i not in te], sorted(te)
rows=[]
for t in TARGETS:
    X,y,scaf=feat[t]; N=len(y)
    for mn,mk in MODELS.items():
        rs=[]
        for s in range(25):
            tr,te=scaffold_split(scaf,N,s)
            if len(te)<5 or len(np.unique(y[te]))<2: continue
            m=mk().fit(X[tr],y[tr]); rs.append(r2_score(y[te],m.predict(X[te])))
        rs=np.array(rs)
        pred=np.zeros(N)
        for tr,te in LeaveOneOut().split(X):
            m=mk().fit(X[tr],y[tr]); pred[te]=m.predict(X[te])
        rows.append({'target':t,'model':mn,'n':N,
                     'scaffold_R2':round(rs.mean(),3),
                     'scaffold_CI':f'[{np.percentile(rs,2.5):.2f},{np.percentile(rs,97.5):.2f}]',
                     'LOO_R2':round(r2_score(y,pred),3),'LOO_RMSE':round(rmse(y,pred),3)})
        print('done:',t,mn)
res=pd.DataFrame(rows); print('\n'+res.to_string(index=False)); res.to_csv('expanded_model_results.csv',index=False)

## Interpretation
* **Landrum is applied uniformly** to the whole ML set — no half-and-half curation.
* Report **both** `scaffold_R2` (honest/novel-chemotype) and `LOO_R2` (standard, usually higher).
* The preclinical split (Part 2) shows how much data the all-phase fetch added over Phase I–IV alone.
* Two targets (5-HT₂A, D3) give an independent read on how the protocol transfers.
* The descriptive chapter is untouched — it stays on the locked `curated_15desc.csv`.